# Rarity-driven sparse retrieval — diagnostics

| Section | What runs | Gate |
|---|---|---|
| Step 1 | `rarity` — the frozen irregular-term list | the top of the list is marked scriptural vocabulary, not tokenizer debris |
| Step 2 | `leakage` — near-duplicate audit, quarantine | few enough flags that quarantining leaves the pool intact |
| Step 3 | `sparse_select` — greedy coverage dry run | full dose ≥ 30% of val queries, else report and stop |


In [ ]:
# e5-large in fp32 over a 10.8k-row pool; any Colab GPU is enough.
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
%cd /home/prnamhr/projects/Style-Aware-MT
!pip install -r requirements.txt

# Text-only pipeline; these two carry an ABI mismatch against the pinned torch.
!pip uninstall -y torchvision torchaudio

`data/knn_index/` is git-ignored, so the pool index is rebuilt each session. The register
centroid is committed and already present.

In [ ]:
!python3 manage.py build_index --config configs/base_qwen.yaml

---
## Step 1 — the rarity list

`rank: surprisal` scores each term with a character 4-gram model fit to the pool
vocabulary weighted by token frequency. Nothing external enters the list.

In [ ]:
!python3 manage.py rarity --config configs/sparse_retrieval.yaml

In [ ]:
import json

import pandas as pd

rarity = json.load(open('results/rarity_train.json'))
cfg = rarity['config']
print(f"{rarity['n_terms']} pool terms, {rarity['n_eligible']} in df "
      f"[{cfg['min_df']}, {cfg['max_df']}] -> {rarity['n_frozen']} frozen "
      f"(requested {cfg['freeze_n']}, rank={cfg['rank']})")
print(f"realized df {rarity['df_observed']}, {rarity['selected_frac']:.1%} of the vocabulary")
print('idf range        :', [round(v, 3) for v in rarity['idf_range']])
print('surprisal range  :', [round(v, 3) for v in rarity['surprisal_range']])
print('pool df histogram  :', rarity['df_histogram']['pool'])
print('frozen df histogram:', rarity['df_histogram']['frozen'])

In [ ]:
# The failure the surprisal rank is meant to fix: an alphabetical tie-break concentrates
# the list on the first few letters of the sort order.
initials = pd.Series([t[0] for t, *_ in rarity['terms']]).value_counts()
print(f'{len(initials)} distinct first letters over {rarity["n_frozen"]} terms; '
      f'the top 5 hold {initials.head(5).sum() / rarity["n_frozen"]:.1%}')
initials.head(12).to_frame('terms').T

In [ ]:
# The check that matters: these should read as marked scriptural vocabulary. If the head
# is broken segmentation instead, the score is ranking tokenizer debris — stop and fix it.
top = pd.read_csv('results/rarity_train_top50.tsv', sep='\t')
top['example'] = top['example'].str.slice(0, 60)
top

### Normalization check — ZWNJ

In [ ]:
collisions = json.load(open('results/rarity_train.json'))['zwnj_collisions']
print(f'{len(collisions)} ZWNJ variant collisions')
pd.DataFrame(collisions, columns=['split spelling', 'joined spelling']).head(25)

---
## Step 2 — leakage audit

In [ ]:
!python manage.py leakage \
    --config configs/sparse_retrieval.yaml \
    --split val \
    --write-quarantine

In [ ]:
leak = json.load(open('results/leakage_val.json'))
print(f"val: {leak['n_eval_rows_flagged']}/{leak['n_eval_rows']} eval rows flagged, "
      f"{leak['n_pool_rows_flagged']} pool rows implicated")
print('  max-cos histogram:', leak['max_cos_histogram'])

pd.DataFrame([
    {'cos': f['cos'], 'jac_src': f['jaccard_source'], 'jac_tgt': f['jaccard_target'],
     'eval': f['eval_source'][:50], 'pool': f['pool_source'][:50]}
    for f in leak['flags'][:10]
])

In [ ]:
!python3 manage.py build_index --config configs/base_qwen.yaml \
    --index_dir data/knn_index_clean \
    --quarantine data/splits/pool_quarantine.json

---
## Step 3 — the sparse channel

The go/no-go. `min_query_terms=1` routes a query to the rarity channel as soon as it
carries one listed term, so the three numbers below are properties of the list, not of
the greedy: how many listed terms a val query carries, how often it carries any, and how
often it carries enough to fill all four rarity slots.

In [ ]:
!python3 manage.py sparse_select --config configs/sparse_retrieval.yaml \
    --split val --index_dir data/knn_index_clean

In [ ]:
sel = json.load(open('results/sparse_selection_val.json'))
print('routes            :', sel['route_fractions'])
print('irregular terms/query — mean', sel['query_terms']['mean'],
      'deciles', sel['query_terms']['deciles'])
print('share at or above :', sel['query_terms']['share_at_or_above'])
print('coverage (routed) :', sel['coverage']['mean'])
print('intra-set cosine  :', sel['intra_set_similarity'])

In [ ]:
# Values from the IDF-ranked list, kept as the thing the new list has to beat.
IDF_RANKED = {'mean matches/query': 0.07, '>=1 match': 0.064, 'full dose (4 slots)': 0.0}
EXPECTED = {'mean matches/query': 3.0, '>=1 match': 0.88, 'full dose (4 slots)': 0.34}
FULL_DOSE_GATE = 0.30

now = {
    'mean matches/query': sel['query_terms']['mean'],
    '>=1 match': sel['query_terms']['share_at_or_above']['1'],
    'full dose (4 slots)': sel['route_fractions']['full'],
}
print(pd.DataFrame({'IDF-ranked': IDF_RANKED, 'now': now, 'expected': EXPECTED}).to_string())

full = now['full dose (4 slots)']
verdict = 'PROCEED' if full >= FULL_DOSE_GATE else 'HOLD'
print(f'\nfull dose {full:.1%} against a {FULL_DOSE_GATE:.0%} gate -> {verdict}')
if verdict == 'HOLD':
    print('Report these numbers; do not generate on this list.')

In [ ]:
import numpy as np
import yaml

from src.retrieval.rarity import load_irregular
from src.retrieval.retrieve import RetrievalIndex
from src.retrieval.sparse import SparseRetriever

CFG = yaml.safe_load(open('configs/sparse_retrieval.yaml'))
RETR, SPA, RAR = CFG['retrieval'], CFG['sparse'], CFG['rarity']
SRC = [json.loads(ln)['input'] for ln in open('data/splits/val.jsonl') if ln.strip()]

index = RetrievalIndex('data/knn_index_clean', embed_model=RETR['embed_model'])
retriever = SparseRetriever(
    index, load_irregular(RAR['out']), index, zwnj=RAR['zwnj'], m=SPA['m'],
    redundancy=SPA['redundancy'], min_query_terms=SPA['min_query_terms'],
)
SELECTED, TRACES = retriever.select_with_trace(SRC, k=RETR['k'])
BASE_SEL = index.retrieve(SRC, k=RETR['k'])

RAR_LEN, COS_LEN = [], []
for t in TRACES:
    rows = set(t['sparse_rows'])
    for r in t['final_rows']:
        (RAR_LEN if r in rows else COS_LEN).append(len(index.pairs[r]['input']))

BASE_CHARS = sum(len(e['input']) for row in BASE_SEL for e in row) / len(TRACES)
ARM_CHARS = (sum(RAR_LEN) + sum(COS_LEN)) / len(TRACES)
for name, v in (('query', [len(x) for x in SRC]), ('rarity channel', RAR_LEN),
                ('cosine fill', COS_LEN)):
    a = np.array(v)
    print(f'{name:16s} n={len(a):6d}  mean {a.mean():6.1f}  median {np.median(a):6.1f} chars')
print(f'exemplar chars per prompt: dense {BASE_CHARS:.0f} -> sparse {ARM_CHARS:.0f} '
      f'({ARM_CHARS / BASE_CHARS - 1:+.1%})')

In [ ]:
for thr in (1, 2, 3, 4):
    print(f'--- min_query_terms={thr}')
    !python3 manage.py sparse_select --config configs/sparse_retrieval.yaml --split val --index_dir data/knn_index_clean --min_query_terms {thr} --out results/sparse_sweep_val_t{thr}.json 2>&1 | grep -E 'routes|intra-set'

### The df band

`min_df` and `max_df` are the two knobs that decide what surprisal is allowed to rank.
Each run writes its own list and report, leaving the configured pair's above intact.

In [ ]:
for min_df, max_df in ((2, 0), (30, 500), (10, 1000), (50, 300)):
    lst = f'results/rarity_train_band{min_df}_{max_df}.json'
    print(f'--- min_df={min_df} max_df={max_df or "inf"}')
    !python3 manage.py rarity --config configs/sparse_retrieval.yaml --min_df {min_df} --max_df {max_df} --out {lst} 2>&1 | grep -E 'frozen|realized|surprisal'
    !python3 manage.py sparse_select --config configs/sparse_retrieval.yaml --split val --index_dir data/knn_index_clean --rarity {lst} --out results/sparse_sweep_val_band{min_df}_{max_df}.json 2>&1 | grep -E 'routes|rarity slots'

### Worked examples

In [ ]:
for ex in sel['examples']:
    print('QUERY :', ex['source'][:90])
    print('  route', ex['trace']['route'], '| terms', ex['trace']['query_terms'],
          '| coverage', ex['trace']['coverage'])
    for e in ex['exemplars']:
        print('   -', e[:90])
    print()